# 02 Polynomial Interpolation

The Lagrange interpolation polynomial is

$$
p_n(x)=\sum_{j=0}^{n} y_j L_j(x),
\qquad
L_j(x)=\prod_{m\ne j}\frac{x-x_m}{x_j-x_m}.
$$

Each basis function satisfies $L_j(x_i)=\delta_{ij}$.

In [ ]:
import pathlib
import sys

import matplotlib.pyplot as plt
import numpy as np

ROOT = pathlib.Path.cwd()
while not (ROOT / "src" / "py_sc").exists() and ROOT != ROOT.parent:
    ROOT = ROOT.parent
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

from py_sc import (
    NaturalCubicSpline,
    chebyshev_nodes,
    lagrange_interpolate,
    newton_divided_differences,
    newton_interpolate,
    piecewise_linear_interpolate,
)

## Lagrange and Newton forms

The Lagrange form is direct and clear. The Newton form uses divided differences:

$$
p_n(x)=c_0+c_1(x-x_0)+c_2(x-x_0)(x-x_1)+\cdots.
$$

Both represent the same polynomial, but Newton form is easier to update when a
new interpolation node is added.

In [ ]:
x = np.array([0.0, 1.0, 2.0, 3.0])
y = np.array([1.0, 2.0, 0.0, 4.0])
x_eval = np.linspace(0.0, 3.0, 200)

lagrange = lagrange_interpolate(x, y, x_eval)
nodes, coefficients = newton_divided_differences(x, y)
newton = newton_interpolate(nodes, coefficients, x_eval)

plt.figure(figsize=(7, 4))
plt.plot(x_eval, lagrange, label="Lagrange")
plt.plot(x_eval, newton, "--", label="Newton")
plt.scatter(x, y, color="black")
plt.legend()
plt.title("Two forms of the same interpolation polynomial")
plt.show()

## Node placement and Runge-type behavior

High-degree interpolation on equally spaced nodes can oscillate near endpoints.
Chebyshev nodes reduce this effect by clustering near the interval boundaries.

In [ ]:
def runge(x):
    return 1.0 / (1.0 + 25.0 * x**2)

x_plot = np.linspace(-1.0, 1.0, 500)
for n in [7, 13]:
    x_equal = np.linspace(-1.0, 1.0, n)
    x_cheb = chebyshev_nodes(-1.0, 1.0, n)
    y_equal = runge(x_equal)
    y_cheb = runge(x_cheb)

    p_equal = lagrange_interpolate(x_equal, y_equal, x_plot)
    p_cheb = lagrange_interpolate(x_cheb, y_cheb, x_plot)

    plt.figure(figsize=(7, 4))
    plt.plot(x_plot, runge(x_plot), color="black", label="Runge function")
    plt.plot(x_plot, p_equal, label=f"equally spaced n={n}")
    plt.plot(x_plot, p_cheb, label=f"Chebyshev n={n}")
    plt.ylim(-0.5, 1.5)
    plt.legend()
    plt.title("Node placement affects polynomial interpolation")
    plt.show()

## Summary

Polynomial interpolation is mathematically elegant, but high-degree global
polynomials require care. Node choice is part of the numerical method, not a
minor implementation detail.